# Experiment 3: CNN on CIFAR-10

This notebook contains the complete runnable source for the experiment. Run all cells to regenerate the report figures and measured results.

In [ ]:
"""Experiment 3: CNN study on CIFAR-10.

The script generates the figures and the measured values used in the report.
It intentionally keeps the architecture small enough to run on a CPU while
still following the sequence prescribed in the laboratory manual.
"""

import json
import os
import pickle
import tarfile
import time
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


ROOT = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
SEED = 42
EPOCHS = 20
BATCH_SIZE = 32
NUM_CLASSES = 10
CLASS_NAMES = [
    "airplane",
    "automobile",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck",
]

np.random.seed(SEED)
tf.random.set_seed(SEED)
try:
    tf.config.threading.set_intra_op_parallelism_threads(4)
    tf.config.threading.set_inter_op_parallelism_threads(2)
except RuntimeError:
    pass


def save_figure(fig, filename):
    fig.tight_layout()
    fig.savefig(ROOT / filename, dpi=110, bbox_inches="tight")
    plt.close(fig)


def build_cnn(pooling="max"):
    Pool = tf.keras.layers.MaxPooling2D if pooling == "max" else tf.keras.layers.AveragePooling2D
    model = tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(32, 32, 3)),
            tf.keras.layers.Conv2D(32, (3, 3), padding="same", activation="relu", name="conv1"),
            Pool(pool_size=(2, 2), name=f"{pooling}_pool1"),
            tf.keras.layers.Conv2D(64, (3, 3), padding="same", activation="relu", name="conv2"),
            Pool(pool_size=(2, 2), name=f"{pooling}_pool2"),
            tf.keras.layers.Flatten(),
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
        ],
        name=f"cifar10_{pooling}_cnn",
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def evaluate_model(model, x_test, y_test, y_test_one_hot):
    probabilities = model.predict(x_test, verbose=0)
    predictions = np.argmax(probabilities, axis=1)
    return {
        "accuracy": float(accuracy_score(y_test, predictions)),
        "precision": float(precision_score(y_test, predictions, average="macro", zero_division=0)),
        "recall": float(recall_score(y_test, predictions, average="macro", zero_division=0)),
        "f1": float(f1_score(y_test, predictions, average="macro", zero_division=0)),
        "predictions": predictions,
        "confusion_matrix": confusion_matrix(y_test, predictions).tolist(),
        "classification_report": classification_report(
            y_test, predictions, target_names=CLASS_NAMES, digits=4, zero_division=0
        ),
        "test_loss": float(model.evaluate(x_test, y_test_one_hot, verbose=0)[0]),
    }


def convolution_output_size(n, f, padding, stride):
    if padding == "same":
        return int(np.ceil(n / stride))
    return int(np.floor((n - f) / stride) + 1)


def load_cifar10():
    """Load CIFAR-10 through Keras, or from a completed local archive."""
    archive = Path.home() / ".keras" / "datasets" / "cifar-10-batches-py-target_archive"
    if archive.exists():
        with tarfile.open(archive, "r:gz") as tar:
            def read_batch(filename):
                member = tar.getmember(f"cifar-10-batches-py/{filename}")
                with tar.extractfile(member) as stream:
                    record = pickle.load(stream, encoding="bytes")
                images = record[b"data"].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1)
                labels = np.asarray(record[b"labels"], dtype="uint8")
                return images, labels

            train_batches = [read_batch(f"data_batch_{index}") for index in range(1, 6)]
            x_train = np.concatenate([batch[0] for batch in train_batches])
            y_train = np.concatenate([batch[1] for batch in train_batches])
            x_test, y_test = read_batch("test_batch")
            return (x_train, y_train), (x_test, y_test)
    return tf.keras.datasets.cifar10.load_data()


def make_dataset_plots(x_train, y_train):
    fig, axes = plt.subplots(2, 5, figsize=(12, 5))
    for index, axis in enumerate(axes.flat):
        axis.imshow(x_train[index])
        axis.set_title(CLASS_NAMES[int(y_train[index])], fontsize=9)
        axis.axis("off")
    fig.suptitle("Sample images from CIFAR-10")
    save_figure(fig, "plot_01_sample_images.png")

    counts = np.bincount(y_train, minlength=NUM_CLASSES)
    fig, axis = plt.subplots(figsize=(8, 4))
    axis.bar(CLASS_NAMES, counts, color="steelblue", edgecolor="black")
    axis.set_title("Class distribution of the CIFAR-10 training set")
    axis.set_xlabel("class")
    axis.set_ylabel("number of images")
    axis.tick_params(axis="x", rotation=30)
    save_figure(fig, "plot_02_class_distribution.png")


def make_history_plots(history):
    fig, axis = plt.subplots(figsize=(9, 5))
    axis.plot(history["accuracy"], label="training accuracy", color="steelblue")
    axis.plot(history["val_accuracy"], label="validation accuracy", color="darkorange")
    axis.set_title("CNN training and validation accuracy")
    axis.set_xlabel("epoch")
    axis.set_ylabel("accuracy")
    axis.legend()
    axis.grid(alpha=0.3)
    save_figure(fig, "plot_03_training_validation_accuracy.png")

    fig, axis = plt.subplots(figsize=(9, 5))
    axis.plot(history["loss"], label="training loss", color="steelblue")
    axis.plot(history["val_loss"], label="validation loss", color="darkorange")
    axis.set_title("CNN training and validation loss")
    axis.set_xlabel("epoch")
    axis.set_ylabel("categorical cross-entropy")
    axis.legend()
    axis.grid(alpha=0.3)
    save_figure(fig, "plot_04_training_validation_loss.png")


def make_feature_map_plot(model, image):
    feature_model = tf.keras.Model(model.inputs[0], model.get_layer("conv1").output)
    feature_maps = feature_model.predict(image[None, ...], verbose=0)[0]
    fig, axes = plt.subplots(2, 4, figsize=(10, 5))
    for index, axis in enumerate(axes.flat):
        fmap = feature_maps[:, :, index]
        axis.imshow(fmap, cmap="viridis")
        axis.set_title(f"filter {index + 1}", fontsize=9)
        axis.axis("off")
    fig.suptitle("Feature maps after the first convolution layer")
    save_figure(fig, "plot_05_feature_maps.png")


def make_confusion_matrix_plot(matrix):
    fig, axis = plt.subplots(figsize=(7, 6))
    sns.heatmap(
        np.asarray(matrix),
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=CLASS_NAMES,
        yticklabels=CLASS_NAMES,
        ax=axis,
    )
    axis.set_title("Confusion matrix of the max-pooling CNN")
    axis.set_xlabel("predicted label")
    axis.set_ylabel("true label")
    save_figure(fig, "plot_06_confusion_matrix.png")


def make_pooling_plot(pool_results):
    names = ["max pooling", "average pooling"]
    accuracy = [pool_results[name]["accuracy"] for name in names]
    fig, axis = plt.subplots(figsize=(7, 4.5))
    bars = axis.bar(names, accuracy, color=["steelblue", "seagreen"], edgecolor="black")
    axis.set_ylim(0, 1)
    axis.set_ylabel("test accuracy")
    axis.set_title("Max pooling versus average pooling")
    for bar, value in zip(bars, accuracy):
        axis.text(bar.get_x() + bar.get_width() / 2, value + 0.01, f"{value:.3f}", ha="center")
    axis.grid(axis="y", alpha=0.3)
    save_figure(fig, "plot_07_pooling_comparison.png")


def make_dimension_plot(dimension_rows):
    labels = [row["label"] for row in dimension_rows]
    values = [row["output"] for row in dimension_rows]
    fig, axis = plt.subplots(figsize=(10, 5))
    bars = axis.bar(labels, values, color="slateblue", edgecolor="black")
    axis.set_ylabel("spatial output dimension")
    axis.set_title("Convolution output dimensions for a 32 x 32 input")
    axis.tick_params(axis="x", rotation=35)
    for bar, value in zip(bars, values):
        axis.text(bar.get_x() + bar.get_width() / 2, value + 0.3, str(value), ha="center", fontsize=8)
    save_figure(fig, "plot_08_convolution_output_dimensions.png")


def main():
    print("Loading CIFAR-10...")
    (x_train, y_train), (x_test, y_test) = load_cifar10()
    y_train = y_train.reshape(-1)
    y_test = y_test.reshape(-1)
    x_train = x_train.astype("float32") / 255.0
    x_test = x_test.astype("float32") / 255.0
    y_train_one_hot = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
    y_test_one_hot = tf.keras.utils.to_categorical(y_test, NUM_CLASSES)

    print("train shape:", x_train.shape, y_train_one_hot.shape)
    print("test shape :", x_test.shape, y_test_one_hot.shape)
    print("pixel range:", float(x_train.min()), "to", float(x_train.max()))
    make_dataset_plots(x_train, y_train)

    dimension_rows = []
    for kernel in (3, 5, 7):
        output = convolution_output_size(32, kernel, "valid", 1)
        dimension_rows.append({"label": f"{kernel}x{kernel}, valid, s=1", "output": output, "shape": f"{output} x {output}"})
    for stride in (1, 2):
        output = convolution_output_size(32, 3, "same", stride)
        dimension_rows.append({"label": f"3x3, same, s={stride}", "output": output, "shape": f"{output} x {output}"})
    for padding in ("same", "valid"):
        output = convolution_output_size(32, 3, padding, 1)
        dimension_rows.append({"label": f"3x3, {padding}, s=1", "output": output, "shape": f"{output} x {output}"})
    make_dimension_plot(dimension_rows)

    pool_results = {}
    histories = {}
    models = {}
    for pooling in ("max", "average"):
        print(f"Training {pooling}-pooling CNN for {EPOCHS} epochs...")
        model = build_cnn(pooling)
        start = time.perf_counter()
        history = model.fit(
            x_train,
            y_train_one_hot,
            validation_split=0.1,
            epochs=EPOCHS,
            batch_size=BATCH_SIZE,
            verbose=2,
        )
        elapsed = time.perf_counter() - start
        evaluation = evaluate_model(model, x_test, y_test, y_test_one_hot)
        evaluation["training_time_seconds"] = elapsed
        evaluation["parameters"] = int(model.count_params())
        evaluation.pop("predictions")
        pool_results[pooling + " pooling"] = evaluation
        histories[pooling] = history.history
        models[pooling] = model
        print(f"{pooling} test accuracy: {evaluation['accuracy']:.4f}")
        print(f"{pooling} training time: {elapsed:.2f} seconds")

    main_model = models["max"]
    main_evaluation = evaluate_model(main_model, x_test, y_test, y_test_one_hot)
    make_history_plots(histories["max"])
    make_feature_map_plot(main_model, x_test[0])
    make_confusion_matrix_plot(main_evaluation["confusion_matrix"])
    make_pooling_plot(pool_results)

    results = {
        "seed": SEED,
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "dataset": {
            "train_shape": list(x_train.shape),
            "test_shape": list(x_test.shape),
            "classes": CLASS_NAMES,
            "pixel_range_after_normalization": [float(x_train.min()), float(x_train.max())],
        },
        "architecture": [
            "Conv2D(32, 3x3, same, ReLU)",
            "MaxPooling2D(2x2)",
            "Conv2D(64, 3x3, same, ReLU)",
            "MaxPooling2D(2x2)",
            "Flatten",
            "Dense(128, ReLU)",
            "Dense(10, Softmax)",
        ],
        "max_pooling": pool_results["max pooling"],
        "average_pooling": pool_results["average pooling"],
        "dimension_rows": dimension_rows,
        "additional_exercises": {
            "64x64_kernel5_stride2_padding2": int(np.floor((64 - 5 + 2 * 2) / 2) + 1),
            "conv_parameters_64_filters_3x3_rgb": int((3 * 3 * 3 + 1) * 64),
        },
        "classification_report": main_evaluation["classification_report"],
    }
    (ROOT / "experiment_3_results.json").write_text(json.dumps(results, indent=2))
    (ROOT / "experiment_3_classification_report.txt").write_text(main_evaluation["classification_report"])
    print("Results written to experiment_3_results.json")


if __name__ == "__main__":
    main()
